# 🦜🔗 LangChain Deep Technical Blog
### Feb 2026 Internship – GenAI Task 2
**Author:** Rakshitha D V  
**Intern ID:** *(your ID here)*  
**Date:** May 2026

---

This notebook contains all working code examples from the LangChain deep technical blog.
Each section maps directly to a section in the Medium article.

## ⚙️ Installation
Run this cell first to install all required packages.

In [ ]:
# Install required libraries
!pip install langchain langchain-community langchain-openai openai faiss-cpu tiktoken -q

## 📦 Imports and API Key Setup

In [ ]:
import os
from dotenv import load_dotenv

# Set your OpenAI API key here (or use a .env file)
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

print("✅ Environment ready")

---
## Section 1 – Basic LLM Call
The simplest possible LangChain usage: send a message, get a reply.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage

# Initialize the chat model
# temperature=0 means deterministic output (good for factual tasks)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Send a basic message
response = llm.invoke([HumanMessage(content="What is LangChain in one sentence?")])

print("LLM Response:")
print(response.content)

---
## Section 2 – Prompt Templates
Instead of hardcoding questions, we use templates with variables.
This makes our prompts reusable and dynamic.

In [ ]:
from langchain.prompts import PromptTemplate

# Define a reusable prompt template
# {topic} is a variable we fill in at runtime
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for a beginner. Keep it under 100 words."
)

# Format the prompt by filling in the variable
formatted_prompt = template.format(topic="vector databases")
print("Formatted Prompt:")
print(formatted_prompt)

# Now send it to the LLM
response = llm.invoke(formatted_prompt)
print("\nLLM Response:")
print(response.content)

---
## Section 3 – Building a Chain (LLMChain)
A chain connects a prompt template directly to an LLM.
This is the core abstraction in LangChain.

In [ ]:
from langchain.chains import LLMChain

# Define the prompt
blog_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Write a short 3-paragraph blog introduction about {topic}. Be engaging."
)

# Create a chain: prompt → LLM → output
blog_chain = LLMChain(llm=llm, prompt=blog_prompt)

# Run the chain
result = blog_chain.invoke({"topic": "the future of AI assistants"})
print(result["text"])

---
## Section 4 – Sequential Chain (Multi-Step Pipeline)
Chain two LLM calls together:
Step 1: Generate a blog title
Step 2: Use that title to write an outline

This shows how chains can pass output forward automatically.

In [ ]:
from langchain.chains import SequentialChain

# Chain 1: Generate a catchy blog title
title_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Generate one catchy blog title about {topic}. Return only the title."
)
title_chain = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

# Chain 2: Use the title to create an outline
outline_prompt = PromptTemplate(
    input_variables=["title"],
    template="Create a 5-point blog outline for the title: '{title}'. Use bullet points."
)
outline_chain = LLMChain(llm=llm, prompt=outline_prompt, output_key="outline")

# Connect both chains sequentially
full_pipeline = SequentialChain(
    chains=[title_chain, outline_chain],
    input_variables=["topic"],
    output_variables=["title", "outline"]
)

# Run the full pipeline
output = full_pipeline.invoke({"topic": "LangChain for beginners"})
print("Generated Title:", output["title"])
print("\nGenerated Outline:\n", output["outline"])

---
## Section 5 – Memory (Conversational Context)
By default, LLMs forget everything after each call.
LangChain Memory fixes this by storing conversation history
and passing it back with every new message.

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

# Create memory that stores the full conversation history
memory = ConversationBufferMemory()

# Create a conversation chain with memory attached
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False  # Set True to see what's happening internally
)

# First message
r1 = conversation.predict(input="Hi! My name is Rakshitha. I'm learning LangChain.")
print("Turn 1:", r1)

# Second message — model should remember the name
r2 = conversation.predict(input="What was my name again?")
print("\nTurn 2:", r2)

# Third message — model should remember the context
r3 = conversation.predict(input="What topic am I learning?")
print("\nTurn 3:", r3)

# Print the stored conversation history
print("\n--- Memory Buffer ---")
print(memory.buffer)

---
## Section 6 – Agents and Tools
Agents let the LLM decide WHICH tool to use based on the question.
Here we give the agent a calculator tool and let it decide when to use it.

In [ ]:
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.tools import tool

# Define a simple calculator tool
@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression. Input should be a valid Python math expression."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Define another tool — a simple lookup
@tool
def get_today_date(query: str) -> str:
    """Returns today's date."""
    from datetime import date
    return str(date.today())

# Bundle tools
tools = [calculator, get_today_date]

# Initialize the agent
# ZERO_SHOT_REACT_DESCRIPTION: agent reasons step-by-step before picking a tool
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True  # Shows the agent's reasoning steps
)

# Ask a question that requires the calculator
response = agent.invoke({"input": "What is 1337 multiplied by 42?"})
print("\nFinal Answer:", response["output"])

---
## Section 7 – Document Loader + Vector Store (RAG Mini Example)
Load a text document, embed it, store in FAISS,
and answer questions about it using similarity search.
This is the foundation of Retrieval Augmented Generation (RAG).

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import CharacterTextSplitter
from langchain.docstore.document import Document
from langchain.chains import RetrievalQA

# Sample document text (replace this with any real text / PDF)
sample_text = """
LangChain is a framework for building LLM-powered applications.
It provides tools for chaining prompts, managing memory, and integrating external tools.
LangChain was created by Harrison Chase and first released in 2022.
It supports multiple LLM providers including OpenAI, Anthropic, HuggingFace, and Cohere.
The main components of LangChain are: LLMs, Prompts, Chains, Memory, Agents, and Tools.
LangChain is widely used in building chatbots, document Q&A systems, and AI agents.
"""

# Step 1: Split text into chunks
splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_text(sample_text)
docs = [Document(page_content=chunk) for chunk in chunks]

print(f"✅ Split into {len(docs)} chunks")

# Step 2: Embed and store in FAISS vector store
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)

print("✅ Vector store created")

# Step 3: Create a RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # 'stuff' = pass all retrieved docs into one prompt
    retriever=vectorstore.as_retriever()
)

# Step 4: Ask a question about the document
question = "Who created LangChain and when?"
answer = qa_chain.invoke({"query": question})
print(f"\nQ: {question}")
print(f"A: {answer['result']}")

---
## ✅ Summary

| Concept | What We Did |
|---|---|
| Basic LLM Call | Sent a message, got a reply |
| Prompt Templates | Made prompts reusable with variables |
| LLMChain | Connected prompt → LLM → output |
| SequentialChain | Chained two LLM calls together |
| Memory | Gave LLM short-term conversation memory |
| Agents + Tools | Let LLM decide which tool to use |
| Vector Store + RAG | Q&A over a custom document |

---
*This notebook is part of the Feb 2026 Internship GenAI Assignment at Innomatics Research Labs.*